# VoyageOps synthetic follow-up QLoRA (free Colab GPU)

This is an optional CUDA exercise for the **free** Colab tier. In **Runtime → Change runtime type**, select a GPU only if Colab offers one without upgrading. Do not select a paid plan or buy compute units. Free GPU availability and session limits vary. The notebook stops immediately if CUDA is unavailable.

It trains a small Qwen2.5 1.5B LoRA adapter on the repository's 60 synthetic training examples, uses 12 for validation, and compares test loss before and after training on 12 held-out examples. This is a pipeline exercise, not proof of improved customer-facing quality. No real traveler data is included; no model or adapter is uploaded.

**Prerequisite:** the current `codex/multi-model-benchmark` branch must be pushed to GitHub; otherwise the notebook will stop before installing packages.

In [ ]:
import torch
assert torch.cuda.is_available(), "No free CUDA GPU available. Stop here; do not upgrade to a paid runtime."
print("GPU:", torch.cuda.get_device_name(0))
print("CUDA:", torch.version.cuda)


In [ ]:
import pathlib, subprocess
repo = pathlib.Path("/content/travel-crm-ai")
if not repo.exists():
    subprocess.run(["git", "clone", "--branch", "codex/multi-model-benchmark", "--single-branch", "https://github.com/Yneq/travel-crm-ai.git", str(repo)], check=True)
script = repo / "scripts/train_followup_cuda.py"
assert script.is_file(), "The CUDA script is not on GitHub yet. Push the local branch first."
subprocess.run(["git", "-C", str(repo), "log", "-1", "--oneline"], check=True)


In [ ]:
subprocess.run(["python", "-m", "pip", "install", "-q", "-r", str(repo / "requirements-colab.txt")], check=True)
print("Optional CUDA training dependencies installed.")


In [ ]:
subprocess.run(["python", str(script), "--data", str(repo / "finetune/followup/data"), "--output", "/content/followup-colab-qlora", "--max-steps", "30"], check=True)


In [ ]:
import json
report = json.loads(pathlib.Path("/content/followup-colab-qlora/report.json").read_text())
print(json.dumps({key: report[key] for key in ("gpu", "steps", "base_test_loss", "adapted_test_loss", "train_loss", "limitations")}, ensure_ascii=False, indent=2))
